# Virginia Piedmont observation and identification timing

This notebook analyzes how quickly observations receive identification engagement from the community in Virginia (statewide).

Current operational definitions:
- `observed_count`: distinct observations grouped by `observed_on`.
- `first_non_owner_identification_delay_days`: days from `observed_on` to the first identification from a community member (non-observer).
- Community responsiveness metric: mean latency until first non-owner ID per observation

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import get_inat_session, get_observation_rows, summarize_time_series

VA_PLACE_ID = 1297  # Virginia statewide place on iNaturalist
# I think we might want a taxon filter here?  At least an iconic-taxon filter. 
START = '2000-01-01'
END = None
PER_PAGE = 200
MAX_PAGES = 40

session = get_inat_session()

In [2]:
# -- Tier 1 preflight: cheap all-time overview + truncation gate --
import plotly.graph_objects as go

# 1. Count-only check (single request, no pagination)
_r = session.get(
    'https://api.inaturalist.org/v1/observations',
    params={'place_id': VA_PLACE_ID, 'd1': START, 'per_page': 0},
)
_total = _r.json().get('total_results', None)
_max_fetchable = PER_PAGE * MAX_PAGES

# 2. All-time weekly histogram (single request, no date filter)
_hist_r = session.get(
    'https://api.inaturalist.org/v1/observations/histogram',
    params={'place_id': VA_PLACE_ID, 'interval': 'week_of_year'},
)
_week_counts = _hist_r.json().get('results', {}).get('week_of_year', {})
_weeks = list(range(1, 53))

# Approximate month starts in week-of-year
_month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

_fig = go.Figure()
_fig.add_scatter(
    x=_weeks,
    y=[_week_counts.get(str(w), 0) for w in _weeks],
    mode='lines',
    name='All observations',
)
_fig.update_layout(
    title='All-time weekly observation volume (Virginia statewide)',
    xaxis=dict(
        tickvals=_month_ticks,
        ticktext=_month_names,
    ),
    yaxis_title='Observations (all time)',
)
_fig.show()

# 3. Truncation gate
print(f'Observations in scope (VA, from {START}): {_total:,}')
print(f'Max fetchable (Tier 2): {_max_fetchable:,}  ({MAX_PAGES} pages x {PER_PAGE}/page)')
print()
if _total is not None and _total > _max_fetchable:
    _pct = _max_fetchable / _total * 100
    print(f'⚠ TRUNCATION WARNING: Tier 2 will fetch only {_pct:.1f}% of observations in scope.')
    print(f'  {_total - _max_fetchable:,} observations will be missing.')
    print()
    print('  Options before running the next cell:')
    print(f'    • Raise MAX_PAGES (need ≥ {-(-_total // PER_PAGE)} to cover all)')
    print(f'    • Narrow START to reduce scope')
    print(f'    • Accept truncation (most recent {_max_fetchable:,} observations)')
else:
    print('✓ Volume OK — Tier 2 pull will capture all observations in scope.')

Observations in scope (VA, from 2000-01-01): 80,498
Max fetchable (Tier 2): 8,000  (40 pages x 200/page)

⚠ TRUNCATION WARNING: Tier 2 will fetch only 9.9% of observations in scope.
  72,498 observations will be missing.

  Options before running the next cell:
    • Raise MAX_PAGES (need ≥ 403 to cover all)
    • Narrow START to reduce scope
    • Accept truncation (most recent 8,000 observations)


In [ ]:
observations = get_observation_rows(
    places=[VA_PLACE_ID],
    d1=START,
    d2=END,
    per_page=PER_PAGE,
    max_pages=MAX_PAGES,
    session=session,
)
observations['observed_on'] = pd.to_datetime(observations['observed_on'])
observations['all_identification_count'] = pd.to_numeric(observations['all_identification_count'], errors='coerce').fillna(0)
observations['non_owner_identification_count'] = pd.to_numeric(observations['non_owner_identification_count'], errors='coerce').fillna(0)
observations['first_non_owner_identification_delay_days'] = pd.to_numeric(observations['first_non_owner_identification_delay_days'], errors='coerce')

# Summary
print(f"Total observations: {len(observations)}")
print(f"Observations with non-owner IDs: {observations['first_non_owner_identification_at'].notna().sum()}")
observations[['obs_id', 'observed_on', 'non_owner_identification_count', 'first_non_owner_identification_delay_days', 'iconic_taxon_name']].head()

Total observations: 8000
Observations with non-owner IDs: 6975


,obs_id,observed_on,non_owner_identification_count,first_non_owner_identification_delay_days,iconic_taxon_name
0,336787503,1916-08-19,1,39998.367095,Insecta
1,143517165,1948-06-26,0,NaN,Insecta
2,144624837,1948-07-01,0,NaN,Insecta
3,148635824,1948-08-08,1,27217.000544,Insecta
4,319983426,1967-04-17,1,21363.974086,Insecta


In [3]:
observed_ts = summarize_time_series(
    observations,
    date_col='observed_on',
    freq='M',
    count_name='observed_count',
    value_aggs={'total_identifications_on_observed_obs': ('all_identification_count', 'sum')},
)

# Time series for observations with non-owner IDs
obs_with_non_owner_ids = observations[observations['first_non_owner_identification_at'].notna()].copy()
obs_with_non_owner_ids['first_non_owner_identification_at'] = pd.to_datetime(
    obs_with_non_owner_ids['first_non_owner_identification_at'], utc=True, errors='coerce'
)

identification_ts = summarize_time_series(
    obs_with_non_owner_ids,
    date_col='first_non_owner_identification_at',
    freq='M',
    count_name='observations_with_non_owner_ids',
    value_aggs={'mean_first_non_owner_delay_days': ('first_non_owner_identification_delay_days', 'mean')},
)

timeline = observed_ts.merge(identification_ts, on='period_start', how='outer').sort_values('period_start')
timeline[['observed_count', 'total_identifications_on_observed_obs', 'observations_with_non_owner_ids']] = timeline[
    ['observed_count', 'total_identifications_on_observed_obs', 'observations_with_non_owner_ids']
].fillna(0)
timeline['pct_with_non_owner_id'] = (timeline['observations_with_non_owner_ids'] / timeline['observed_count'].replace(0, np.nan)) * 100
timeline.tail()

c:\Users\drsvs\Desktop\code\pynat\helpers.py:853: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  working['period_start'] = working[date_col].dt.to_period(freq).dt.to_timestamp()
c:\Users\drsvs\Desktop\code\pynat\helpers.py:853: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  working['period_start'] = working[date_col].dt.to_period(freq).dt.to_timestamp()


,period_start,observed_count,total_identifications_on_observed_obs,observations_with_non_owner_ids,mean_first_non_owner_delay_days,pct_with_non_owner_id
357,2025-11-01,0.0,0.0,26.0,6986.370040,NaN
358,2025-12-01,0.0,0.0,29.0,6452.197419,NaN
359,2026-01-01,0.0,0.0,13.0,5981.073721,NaN
360,2026-02-01,0.0,0.0,27.0,7353.260999,NaN
361,2026-03-01,0.0,0.0,49.0,5835.724980,NaN


In [4]:
counts_plot = timeline.melt(
    id_vars='period_start',
    value_vars=['observed_count', 'observations_with_non_owner_ids'],
    var_name='series',
    value_name='count',
)

fig = px.line(
    counts_plot,
    x='period_start',
    y='count',
    color='series',
    markers=True,
    title='Observation volume vs non-owner identification volume',
)
fig.show()

ratio_plot = timeline[['period_start', 'mean_first_non_owner_delay_days', 'pct_with_non_owner_id']].copy()
ratio_plot = ratio_plot.dropna(subset=['mean_first_non_owner_delay_days'])

ratio_fig = px.scatter(
    ratio_plot,
    x='period_start',
    y='mean_first_non_owner_delay_days',
    size='pct_with_non_owner_id',
    markers=True,
    title='Mean latency to first non-owner identification (bubble size = % of monthly observations)',
)
ratio_fig.show()

TypeError: scatter() got an unexpected keyword argument 'markers'

In [ ]:
delay_by_taxon = (
    observations.dropna(subset=['first_non_owner_identification_delay_days'])
    .groupby('iconic_taxon_name', dropna=False)
    .agg(
        observations_with_non_owner_ids=('obs_id', 'size'),
        mean_delay_days=('first_non_owner_identification_delay_days', 'mean'),
        median_delay_days=('first_non_owner_identification_delay_days', 'median'),
        min_delay_days=('first_non_owner_identification_delay_days', 'min'),
        max_delay_days=('first_non_owner_identification_delay_days', 'max'),
    )
    .reset_index()
    .sort_values(['observations_with_non_owner_ids', 'mean_delay_days'], ascending=[False, True])
)
delay_by_taxon

,iconic_taxon_name,identification_event_count,observations_represented,mean_delay_days,median_delay_days
4,Aves,7433,2751,41.899154,1.122986
10,Plantae,3977,1989,56.151270,1.281678
6,Fungi,1838,1097,80.084742,1.587847
7,Insecta,1810,744,78.302670,1.510243
8,Mammalia,1275,435,28.565514,1.323391
1,Amphibia,1052,353,11.381241,1.767448
2,Animalia,419,214,142.642979,1.660683
3,Arachnida,410,176,77.299301,1.316933
12,Reptilia,288,75,77.925018,1.749479
9,Mollusca,152,82,106.828057,1.378113


In [ ]:
obs_for_timeline = observations.dropna(subset=['first_non_owner_identification_delay_days']).copy()
obs_for_timeline['first_non_owner_identification_at'] = pd.to_datetime(
    obs_for_timeline['first_non_owner_identification_at'], utc=True, errors='coerce'
)

delay_timeline = summarize_time_series(
    obs_for_timeline,
    date_col='first_non_owner_identification_at',
    freq='M',
    group_cols=['iconic_taxon_name'],
    count_col='obs_id',
    count_name='obs_count',
    value_aggs={'mean_delay_days': ('first_non_owner_identification_delay_days', 'mean')},
)
delay_timeline = delay_timeline[delay_timeline['obs_count'] >= 25].copy()

delay_fig = px.line(
    delay_timeline,
    x='period_start',
    y='mean_delay_days',
    color='iconic_taxon_name',
    markers=True,
    title='Mean latency to first non-owner identification by taxon (≥25 obs/month)',
)
delay_fig.show()

c:\Users\drsvs\Desktop\code\pynat\helpers.py:853: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  working['period_start'] = working[date_col].dt.to_period(freq).dt.to_timestamp()
